In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","") 
    return text.split()

In [4]:
tokenize("What is the capitals of france?")

['what', 'is', 'the', 'capitals', 'of', 'france']

In [5]:
# vocab
vocab = {'UNK':0}

In [6]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer


    for token in merged_tokens:

        if token not in vocab:
            vocab[token] = len(vocab)


In [7]:
build_vocab(df.iloc[0])

In [8]:
vocab

{'UNK': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7}

In [9]:
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
88    None
89    None
90    None
91    None
92    None
Length: 93, dtype: object

In [10]:
len(vocab)

334

In [11]:
#convert words to numbers
def text_to_indices(text,vocab):

    indexed_test = []

    for token in tokenize(text):
        
        if token in vocab:
            indexed_test.append(vocab[token])

        else:
            indexed_test.append(vocab['UNK'])

    return indexed_test

In [12]:
text_to_indices('what is the capital of france',vocab)

[1, 2, 3, 4, 5, 6]

In [13]:
import torch 
from torch.utils.data import Dataset, DataLoader

In [14]:
class QADataSet(Dataset):

    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

        

In [15]:
dataset = QADataSet(df,vocab)

In [16]:
dataloader  = DataLoader(dataset,batch_size=1, shuffle=True)

In [17]:
for question , answer in dataloader:
    print(question[0],answer[0])
    #print(question,answer)

tensor([  1,   2,   3, 141, 117,  83,   3, 277, 278]) tensor([121])
tensor([  1,   2,   3,   4,   5, 113]) tensor([114])
tensor([ 42, 290, 291, 118, 292, 158, 293, 294]) tensor([295])
tensor([ 42, 312,   2, 313,  62,  63,   3, 314, 315]) tensor([316])
tensor([ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]) tensor([273])
tensor([ 1,  2,  3, 37, 38, 39, 40]) tensor([41])
tensor([  1,   2,   3, 234,   5, 235]) tensor([131])
tensor([ 10,  96,   3, 104, 239]) tensor([240])
tensor([ 10,  75, 111]) tensor([112])
tensor([ 1,  2,  3, 50, 51, 19,  3, 45]) tensor([52])
tensor([ 10,  29, 130, 131]) tensor([132])
tensor([ 42,   2,   3, 210, 137, 168, 211, 169]) tensor([113])
tensor([ 1,  2,  3, 69,  5, 53]) tensor([260])
tensor([  1,   2,   3, 103,   5, 104,  19, 105]) tensor([106])
tensor([  1,   2,   3,   4,   5, 286]) tensor([287])
tensor([ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]) tensor([145])
tensor([  1,   2,   3,  17, 115,  83,  84]) tensor([116])
tensor([  1,   2,   3, 212,   5,

In [18]:
from torch import nn 

In [ ]:
# not using sequencial model because sequencial expects only one output from the each layer but rnn generates two outputs  
class MySimpleRNN(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.RNN(50,64,batch_first=True)
        self.fc = nn.Linear(64,vocab_size)

    def forward(self,question):
        embedded_question = self.embedding(question)
        hidden,final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))


        return output

In [20]:
# Not training on GPU as Dataset is small 
model = MySimpleRNN(len(vocab))

In [21]:
learning_rate = 0.001
epochs = 25

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [23]:
for epoch in range(epochs):
    total_loss = 0 
    for question , answer in dataloader:
        optimizer.zero_grad()

        output = model(question)

        loss = criterion(output,answer[0])

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch :{epoch + 1 } , Loss: {total_loss:4f}")

Epoch :1 , Loss: 541.961603
Epoch :2 , Loss: 469.045132
Epoch :3 , Loss: 387.903562
Epoch :4 , Loss: 324.989342
Epoch :5 , Loss: 272.275833
Epoch :6 , Loss: 223.541146
Epoch :7 , Loss: 180.600742
Epoch :8 , Loss: 142.332379
Epoch :9 , Loss: 111.111512
Epoch :10 , Loss: 86.731239
Epoch :11 , Loss: 68.138898
Epoch :12 , Loss: 55.015118
Epoch :13 , Loss: 44.994156
Epoch :14 , Loss: 36.976793
Epoch :15 , Loss: 31.126920
Epoch :16 , Loss: 26.389548
Epoch :17 , Loss: 22.407738
Epoch :18 , Loss: 19.348716
Epoch :19 , Loss: 16.707307
Epoch :20 , Loss: 14.439666
Epoch :21 , Loss: 12.527260
Epoch :22 , Loss: 10.926773
Epoch :23 , Loss: 9.610613
Epoch :24 , Loss: 8.487038
Epoch :25 , Loss: 7.577979


In [24]:
def predict(model , question, threshold=0.5):
    # convert question to numbers
    numerical_question = text_to_indices(question, vocab)

    # tensor
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    output = model(question_tensor)

    # convert logits to probs
    probs = torch.nn.functional.softmax(output, dim=1)

    value, index = torch.max(probs, dim=1)

    if value < threshold:
        print("I don't know")

    print(list(vocab.keys())[index])

In [40]:
predict(model,"who founded electricity?")

benjamin-franklin
